# Analysis of the Conservation State of Ancient Books

**Kimia — Information Systems and Business Intelligence** — A.Y. 2025/2026

Run all cells after cloning the repo. I focus on humidity and parchment/iron-gall when explaining results.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

def _run(cmd):
    subprocess.run(cmd, shell=True, check=False)

# Colab / Jupyter: clone repo if src/ is missing
if not Path('src').exists():
    _run('git clone https://github.com/kimias21/book-conservation-bi.git')
    os.chdir('book-conservation-bi')

PROJECT_ROOT = Path('.').resolve()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

_run('pip install -q -r requirements.txt')
if not Path('data/processed/integrated_heritage_books.csv').exists():
    _run(f'{sys.executable} -m src.data_integration')

import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.config import INTEGRATED_CSV, DATA_PROCESSED
from src.conservation_index import add_conservation_columns

sns.set_theme(style='whitegrid')
print('Project root:', PROJECT_ROOT)
print('Data file exists:', INTEGRATED_CSV.exists())


## 1. Loading and pre-processing

In [ ]:
df = pd.read_csv(INTEGRATED_CSV)
print(df.shape)
df.head()

In [ ]:
TAXONOMY = {'parchment':'Parchment','laid_paper':'Laid paper','industrial_paper':'Industrial paper','mixed':'Mixed support'}
df['support_material_label'] = df['support_material'].map(TAXONOMY)
for c in ['flood_event','hvac_failure_event','restoration_recent']:
    df[c] = df[c].astype(bool)
print('Missing:', df.isnull().sum().sum())
df.describe().T

After loading 480 volumes I check humidity first — in my data Bologna (Archiginnasio) has the highest mean CRI because RH is often above 55%.


## 2. Exploratory analysis

In [ ]:
df['conservation_risk_index'].hist(bins=30)
plt.title('CRI distribution'); plt.show()
sns.boxplot(data=df, x='support_material_label', y='conservation_risk_index')
plt.xticks(rotation=30); plt.show()
sns.heatmap(df[['conservation_risk_index','avg_relative_humidity_pct','avg_temperature_c','age_stress','material_stress']].corr(), annot=True)
plt.show()

## 3. Conservation Risk Index

In [ ]:
df = add_conservation_columns(df)
print(df['risk_level'].value_counts())
print(df[['conservation_risk_index','observed_condition_score']].corr())

## 4. Models

In [ ]:
groups = [g['conservation_risk_index'].values for _, g in df.groupby('support_material')]
print('ANOVA F,p:', stats.f_oneway(*groups))

# Target: elevated risk (CRI >= 45) — more stable than high_risk with few positives
df['elevated_risk'] = (df['conservation_risk_index'] >= 45).astype(int)
print('Elevated risk count:', df['elevated_risk'].sum())

feature_cols = ['century','avg_temperature_c','avg_relative_humidity_pct','avg_light_lux',
                'air_quality_index','support_material','ink_type','binding_type','site_type']
X, y = df[feature_cols], df['elevated_risk']
cat = ['support_material','ink_type','binding_type','site_type']
num = [c for c in feature_cols if c not in cat]
prep = ColumnTransformer([('num', StandardScaler(), num),
                          ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])
split_args = dict(test_size=0.25, random_state=42)
if y.sum() >= 10 and (len(y) - y.sum()) >= 10:
    split_args['stratify'] = y
X_train, X_test, y_train, y_test = train_test_split(X, y, **split_args)
rf = Pipeline([('prep', prep),
               ('clf', RandomForestClassifier(200, random_state=42, class_weight='balanced'))])
rf.fit(X_train, y_train)
print(classification_report(y_test, rf.predict(X_test), zero_division=0))


### Geographic and temporal visualisations

In [ ]:
ts = pd.read_csv(DATA_PROCESSED / 'environmental_timeseries.csv')
fig, ax = plt.subplots(figsize=(10,4))
for sid, g in ts.groupby("site_id"):
    ax.plot(g["month"], g["avg_relative_humidity_pct"], label=sid)
ax.set_ylabel("RH %"); ax.set_title("Temporal evolution of relative humidity")
ax.legend(ncol=4, fontsize=7); plt.xticks(rotation=45); plt.tight_layout(); plt.show()


### Clustering risk profiles

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
X = df[["environmental_stress","material_stress","age_stress","event_stress"]]
km = KMeans(4, random_state=42, n_init=10).fit(StandardScaler().fit_transform(X))
df["cluster"] = km.labels_
print(df.groupby("cluster")["conservation_risk_index"].mean())


## 5. Discussion
See report/final_report.md for decision-maker narrative, assumptions, and limitations.